# ReceiptIQ - Google Colab Setup & Benchmarking

This notebook sets up ReceiptIQ in Google Colab, initializes the database, processes the CORD dataset, and runs comprehensive benchmarks comparing different model configurations.

## 1. Clone Repository

In [ ]:
import os
import subprocess

# Clone repository
if not os.path.exists('/content/ReceiptIQ'):
    !git clone https://github.com/your-org/ReceiptIQ.git /content/ReceiptIQ
    os.chdir('/content/ReceiptIQ')
else:
    os.chdir('/content/ReceiptIQ')
    print("Repository already cloned")

print("✓ Repository ready")
print(f"Current directory: {os.getcwd()}")

## 2. Install System Dependencies

In [ ]:
!apt-get update -qq > /dev/null
!apt-get install -qq -y tesseract-ocr poppler-utils 2>&1 | grep -E "Setting up|done"
print("✓ System dependencies installed (tesseract-ocr, poppler-utils)")

## 3. Install Python Requirements

In [ ]:
!pip install -q -r requirements.txt
!pip install -q datasets torch torchvision transformers

print("✓ Python dependencies installed")

## 4. Initialize Database

In [ ]:
from app.tools.db import init_db

# Initialize database
try:
    init_db()
    print("✓ Database initialized successfully")
except Exception as e:
    print(f"⚠ Database initialization: {e}")
    print("  (May already exist - continuing)")

## 5. Download and Process CORD Dataset

In [ ]:
import subprocess
import os

os.chdir('/content/ReceiptIQ')

# Download CORD 100 subset
print("Downloading CORD 100 dataset...")
!python scripts/download_cord_subset.py

print("\nFiltering clean CORD data...")
!python scripts/filter_clean_cord.py

print("✓ CORD dataset downloaded and processed")

## 6. Ingest Data to Database

In [ ]:
os.chdir('/content/ReceiptIQ')

print("Ingesting CORD data into database...")
!python scripts/load_images_to_db.py

print("✓ Data ingestion complete")

## 7. Launch Gradio UI

In [ ]:
import gradio as gr
from app.main import create_ui

print("Launching Gradio UI...")
ui = create_ui()
ui.launch(
    share=True,
    debug=False,
    server_name="0.0.0.0",
    server_port=7860,
)
print("✓ Gradio UI is running")

## 8. Run Benchmark Tests

Running comprehensive benchmarks to compare model configurations...

In [ ]:
import json
import pandas as pd

os.chdir('/content/ReceiptIQ')

# Store results for comparison
benchmark_results = {}

# Benchmark 1: phi_only with cache ON
print("=" * 70)
print("Benchmark 1/4: phi_only with cache ON")
print("=" * 70)
!python scripts/run_benchmark.py --model_mode phi_only --cache on --prompt_cache on

benchmark_results['phi_only_cache_on'] = "Completed"
print("✓ Benchmark 1 complete\n")

In [ ]:
# Benchmark 2: phi_only with cache OFF
print("=" * 70)
print("Benchmark 2/4: phi_only with cache OFF")
print("=" * 70)
!python scripts/run_benchmark.py --model_mode phi_only --cache off --prompt_cache off

benchmark_results['phi_only_cache_off'] = "Completed"
print("✓ Benchmark 2 complete\n")

In [ ]:
# Benchmark 3: phi+mistral with cache ON
print("=" * 70)
print("Benchmark 3/4: phi+mistral with cache ON")
print("=" * 70)
!python scripts/run_benchmark.py --model_mode phi+mistral --cache on --prompt_cache on

benchmark_results['phi_mistral_cache_on'] = "Completed"
print("✓ Benchmark 3 complete\n")

In [ ]:
# Benchmark 4: phi+mistral with cache OFF
print("=" * 70)
print("Benchmark 4/4: phi+mistral with cache OFF")
print("=" * 70)
!python scripts/run_benchmark.py --model_mode phi+mistral --cache off --prompt_cache off

benchmark_results['phi_mistral_cache_off'] = "Completed"
print("✓ Benchmark 4 complete\n")

print("=" * 70)
print("✓ ALL BENCHMARKS COMPLETED")
print("=" * 70)

## 9. Display Results Table

In [ ]:
import glob
from pathlib import Path

# Load the most recent benchmark results
output_dir = Path('/content/ReceiptIQ/outputs')
json_files = sorted(glob.glob(str(output_dir / '*summary*.json')), reverse=True)

if json_files:
    latest_file = json_files[0]
    with open(latest_file, 'r') as f:
        data = json.load(f)
    
    # Extract statistics
    stats_list = []
    config = data.get('config', {})
    
    for model_mode in ['phi_only', 'phi+mistral']:
        for cache_status in ['on', 'off']:
            for prompt_cache in ['on', 'off']:
                # Find matching stats
                if (config.get('model_mode') == model_mode and 
                    config.get('cache_enabled') == (cache_status == 'on') and
                    config.get('prompt_cache_enabled') == (prompt_cache == 'on')):
                    
                    stats = data.get('statistics', {})
                    stats_list.append({
                        'Model': model_mode,
                        'Cache': cache_status,
                        'Prompt Cache': prompt_cache,
                        'Avg Latency (ms)': f"{stats.get('avg_latency_ms', 0):.1f}",
                        'Success Rate (%)': f"{stats.get('overall_success_rate', 0):.1f}",
                        'Citation Rate (%)': f"{stats.get('citation_rate', 0):.1f}",
                    })
    
    # Create and display results table
    if stats_list:
        results_df = pd.DataFrame(stats_list)
        print("\n" + "=" * 80)
        print("BENCHMARK RESULTS COMPARISON")
        print("=" * 80)
        print(results_df.to_string(index=False))
        print("=" * 80)
    else:
        print("Loading benchmark results from output files...")
        # Load all summary files for comprehensive view
        print(f"Found {len(json_files)} benchmark result files")
        for f in json_files[:4]:
            print(f"  • {Path(f).name}")
else:
    print("⚠ No benchmark result files found yet")
    print("Run the benchmark cells above to generate results")